# Calculate G & S from .ptu files

## Dependencies

Type the following commands in your conda command prompt (with the correct conda environment active!) to install the dependency packages.

- conda install numba
- conda install pandas
- conda install scipy
- pip install -U "ptufile[all]"

Download the indicated files from the indicated github repository and place the files in the same folder as this script.
- Repository: https://github.com/jayunruh/FLIM_tools | Files: flimtools.py  & linleastsquares.py

In [1]:
import numpy as np
import pandas as pd
import flimtools
import os
import re
import ptufile

## Select folder

- The `in_dir` folder is assumed to contain the ptu files as input


In [2]:
in_dir = 'G-Ca-FLITS_Calibration.sptw/'

## Read the files, calculate G&S and save

We first find the bin of photon arrival time in which the highest number of photons were detected, in order to find the relative location of our laser pulse

In [3]:
# Read a list of files from in_dir
filelist = os.listdir(in_dir)
# Initialize a list to hold all peak bins in degrees
all_peaks = []
# Initialize a list to hold the column (i.e. sample) names
header = []

for file in filelist:
    if file.lower().endswith('.ptu') is True:
        
        ###################### from ptu to FLIM stack #############
        # Convert each .put file to a FLIM stack
        ptu_file  = ptufile.imread(in_dir+file, dtime=0)
        FLIM_stack = ptu_file[0, :, :, 0, :]
        #transpose the data to list the time dimension first
        FLIM_stack = np.transpose(FLIM_stack, (2, 0, 1))
        ######################
        
        
        ###################### average decay #############
        # Sum each time bin in the stack to get a single decay from the stack
        # Store this data in a dataframe for every image/condition
        
        #Sum the pixels for each time point
        decay_profile = FLIM_stack.sum(axis=(1, 2))
        #Remove reflection effect by removing all frames after 200
        decay_profile = decay_profile[:201]
        # Create the column name
        sample = file.replace('.ptu', '')

        # Find peak intensity bin of decay profile and append to dataframe for all samples
        peak_bin_decay = np.argmax(decay_profile[0:20])
        all_peaks.append(peak_bin_decay)
        header.append(sample)


# Create a DataFrame from the list of peak bins
df_peaks = pd.DataFrame(all_peaks).transpose()

# Set the column names of the DataFrame
df_peaks.columns = header

print(df_peaks)

   F11  G10  G11  G2  G3  G4  G5  G6  G7  G8  G9
0   15   15   15  16  16  16  16  16  16  15  15


We then set a global peak bin value that will be used to calculate G & S for all samples

In [4]:
#The values in df_peaks should all be very similar, this then simply calculates the in-between point for all these samples to use as a "global peak" so that each sample's period is aligned
global_peak_bin_decay = np.mean(df_peaks)
global_deg_peak_bin_decay = global_peak_bin_decay/len(decay_profile)*360
print(global_deg_peak_bin_decay)

27.84260515603799


Finally we calculate the average G&S for each image using the global peak bin value

In [5]:
# Initialize a list to hold the summed pixels arrays
all_decays = []
all_decays_raw = []
# Initialize a list to hold the column names
header = []

# Initialize an empty list to collect DataFrames
df_GS = []

for file in filelist:
    if file.lower().endswith('.ptu') is True:
        
        ###################### from ptu to FLIM stack #############
        # Convert each .put file to a FLIM stack
        ptu_file  = ptufile.imread(in_dir+file, dtime=0)
        FLIM_stack = ptu_file[0, :, :, 0, :]
        #transpose the data to list the time dimension first
        FLIM_stack = np.transpose(FLIM_stack, (2, 0, 1))
        ######################
        
        
        ###################### average decay #############
        # Sum each time bin in the stack to get a single decay from the stack
        # Store this data in a dataframe for every image/condition
        
        #Sum the pixels for each time point and save the unmodified decay profile
        decay_profile = FLIM_stack.sum(axis=(1, 2))
        all_decays_raw.append(decay_profile)
        #Remove reflection effect by removing all frames after 200
        decay_profile = decay_profile[:201]
        # Create the column name
        sample = file.replace('.ptu', '')
        # Use regular expression to find patterns like _t1, _t2, ..., _t23, and replace them with _t001, _t002, ..., _t023
        sample = re.sub(r'_t(\d{1,2})$', lambda x: f'_t{x.group(1).zfill(3)}', sample)
        # Append the summed pixels to the list
        all_decays.append(decay_profile)
        header.append(sample)
        #######################
        
        
        ####################### average G,S, Intensity #############
        # Calculate the average S and G coordinates from the decay
        avg_G, avg_S = flimtools.calcGS(decay_profile,global_deg_peak_bin_decay,1.0)
        # Create a DataFrame for the new row and store this data in a dataframe for every image/condition
        new_row_df = pd.DataFrame({
            'sample': [sample],
            'avg_G': [avg_G],
            'avg_S': [avg_S],
            'Int': [decay_profile.sum()],
        })
        df_GS.append(new_row_df)
        #######################

        
# Create a DataFrame from the list of summed pixels arrays
df_decays = pd.DataFrame(all_decays).transpose()
df_raw_decays = pd.DataFrame(all_decays_raw).transpose()
# Set the column names of the DataFrame
df_decays.columns = header
df_raw_decays.columns = header        
# Wirte decay dataframes to .csv files for visual inspection
df_decays.to_csv('all_decays.csv', index=True)
df_raw_decays.to_csv('all_decays_raw.csv', index=True)

# Write the G&S DataFrame to a CSV file
df_GS_all = pd.concat(df_GS, ignore_index=True)
print(df_GS_all)
df_GS_all.to_csv('df_GS.csv', index=False)
 

        
print('\nDone!')


   sample     avg_G     avg_S       Int
0     F11  0.623947  0.381809   7738046
1     G10  0.575427  0.391550   8402222
2     G11  0.603613  0.387479   9564704
3      G2  0.386473  0.431486  16501830
4      G3  0.388407  0.431133  13802172
5      G4  0.400190  0.429317  12352517
6      G5  0.413973  0.425981  11392104
7      G6  0.440907  0.420263  10094966
8      G7  0.484904  0.410757   8971440
9      G8  0.526139  0.401875   8170595
10     G9  0.556934  0.395082   7669661

Done!
